# Zaskaleta AI Twin — AUTO v4
One-click production with visible installer logs and resume from Google Drive.

**From you:** T4 GPU → Run all → approve Google Drive access once.


In [ ]:
import os, re, subprocess, sys
from pathlib import Path
import torch
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Увімкніть T4 GPU: Runtime → Change runtime type → T4 GPU')

from google.colab import drive
drive.mount('/content/drive')

ROOT=Path('/content/zaskaleta-ai-twin-colab')
if ROOT.exists():
    subprocess.run(['rm','-rf',str(ROOT)],check=True)
subprocess.run(['git','clone','--depth','1','https://github.com/sergokharkov/zaskaleta-ai-twin-colab.git',str(ROOT)],check=True)

WORKER=ROOT/'worker'
env=os.environ.copy()
env['APP_DIR']=str(WORKER)
env['MUSETALK_ROOT']='/content/MuseTalk'
env['VENV_DIR']='/content/ai-twin-py311'

print('\n========== INSTALLER START ==========')
iproc=subprocess.Popen(['bash',str(WORKER/'install_gpu_engines.sh')],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
install_lines=[]
for line in iproc.stdout:
    print(line,end='')
    install_lines.append(line)
icode=iproc.wait()
if icode != 0:
    tail=''.join(install_lines[-80:])
    print('\n❌ INSTALLER FAILED — last output:\n'+tail)
    raise RuntimeError(f'Installer stopped with exit code {icode}')
print('========== INSTALLER OK ==========\n')

PY='/content/ai-twin-py311/bin/python'
cmd=[PY,str(WORKER/'run_auto_v3.py'),'--root',str(ROOT),'--mydrive','/content/drive/MyDrive']
proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
lines=[]
for line in proc.stdout:
    print(line,end='')
    lines.append(line)
code=proc.wait()
if code != 0:
    print('\n❌ AUTO PIPELINE FAILED — last output:\n'+''.join(lines[-80:]))
    raise RuntimeError(f'AUTO v4 stopped with exit code {code}')
out=''.join(lines)
m=re.search(r'^FINAL_PATH=(.+)$',out,re.MULTILINE)
if not m:
    raise RuntimeError('AUTO v4 завершився без FINAL_PATH')
FINAL=m.group(1).strip()

from IPython.display import Video, display
print('✅ FINAL:',FINAL)
display(Video(FINAL,embed=True,width=360))
